# CodeNet 4L — Small Graph Inspection

Source-release and final graph-export sanity checks.


In [ ]:
# SPECTRA notebook progress: begin
from tqdm.auto import tqdm

def _spectra_enable_cell_progress():
    from IPython import get_ipython
    _shell = get_ipython()
    if _shell is None:
        return
    _old = getattr(_shell, "_spectra_cell_progress", None)
    if _old is not None:
        _old[0].close()
        for _event, _callback in _old[1:]:
            _shell.events.unregister(_event, _callback)
    _bar = tqdm(total=4, desc="Notebook cells", unit="cell", mininterval=2)
    def _after_cell(_result):
        if _bar.n < _bar.total:
            _bar.update(1)
        if getattr(_result, "error_in_exec", None) or getattr(_result, "error_before_exec", None):
            _bar.set_description("Notebook cell failed")
            _bar.close()
        elif _bar.n >= _bar.total:
            _bar.close()
    _shell.events.register("post_run_cell", _after_cell)
    _shell._spectra_cell_progress = (_bar, ("post_run_cell", _after_cell))

_spectra_enable_cell_progress()
# SPECTRA notebook progress: end

from pathlib import Path
import sys
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'spectral_code').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook from the repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from spectral_code.evaluation.codenet_analysis import *

sns.set_theme(style='whitegrid', context='notebook')
ROOT = clean_data_dir()
ROOT


## Source release


In [ ]:
release = source_release_summary()
pd.DataFrame(release['pair_buckets'])


## Graph readiness


In [ ]:
coverage, summary = graph_coverage(ROOT)
display(summary)
display(coverage)


In [ ]:
codes = load_codes(ROOT, include_code=True)
codes[['code_id', 'source_code_id', 'language', 'problem_id', 'is_mutant', 'line_count', 'code']].sample(3, random_state=42)
